In [1]:
from dotenv import load_dotenv
import os
from langchain_ollama import ChatOllama, OllamaEmbeddings
import arxiv
import pymupdf4llm
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language, MarkdownHeaderTextSplitter
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.chains.history_aware_retriever import create_history_aware_retriever
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_core.messages import HumanMessage, AIMessage

/home/fahad/projects/PaperPilot/backend/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_1753/1085157836.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


In [2]:
llm = ChatOllama(
    model = "llama3.2:3b"
)

In [3]:
llm.invoke("Hey")

AIMessage(content='How can I assist you today?', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-07-26T15:32:18.563823645Z', 'done': True, 'done_reason': 'stop', 'total_duration': 12977914088, 'load_duration': 12485398683, 'prompt_eval_count': 26, 'prompt_eval_duration': 376712000, 'eval_count': 8, 'eval_duration': 108020000, 'logprobs': None, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama'}, id='lc_run--019f9f0e-0b49-72a0-b3f0-bc0370b80e29-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 26, 'output_tokens': 8, 'total_tokens': 34})

In [4]:
embedding_model = OllamaEmbeddings(
    model = "nomic-embed-text",
)

In [5]:
embedding_model.embed_query("Hsad")

[0.030157706,
 0.004875144,
 -0.1394687,
 0.01727163,
 0.0035628334,
 0.050416097,
 0.056316845,
 0.07400536,
 0.058664043,
 0.011401007,
 0.029788699,
 0.04796992,
 0.10067217,
 -0.023540258,
 0.018963346,
 0.029449612,
 -0.047006525,
 0.027459176,
 -0.053250883,
 -0.016283037,
 -0.057408903,
 0.0018994301,
 -0.048666295,
 -0.02968504,
 0.11942874,
 -0.008939691,
 -0.061933953,
 -0.037849426,
 0.016372383,
 0.0017718511,
 -0.011169176,
 -0.037982732,
 0.009525859,
 -0.0075244345,
 -0.005578565,
 -0.0056474516,
 0.035364807,
 0.039997354,
 -0.04338477,
 0.015497923,
 0.011997303,
 -0.047789168,
 -0.003001044,
 -0.09226196,
 0.022433162,
 -0.003263958,
 0.034630265,
 -0.038882393,
 0.07758411,
 0.038525674,
 -0.0309727,
 -0.023189904,
 -0.03354293,
 -0.055944085,
 0.021156218,
 -0.016704233,
 -0.0036087474,
 -0.04428689,
 0.012601456,
 -0.04737844,
 0.047300894,
 0.06526979,
 -0.06864407,
 0.033238888,
 0.0029666503,
 -0.007648285,
 -0.012881404,
 0.042578667,
 -0.024160294,
 -0.0099705

In [6]:
client = arxiv.Client()
search = arxiv.Search(
    query="BMI",
    max_results=4,
    sort_by=arxiv.SortCriterion.SubmittedDate
)

In [7]:
papers = []
for result in client.results(search):
    papers.append({
        "title": result.title.replace("\n", " "),
        "authors": [author.name for author in result.authors],
        "published": result.published.strftime("%Y-%m-%d"),
        "summary": result.summary.replace("\n", " ")[:800] + "...",
        "url": result.entry_id
    })


In [8]:
"""Formats raw ArXiv paper data into clean Markdown links."""
# if not papers:
#     return "No recent papers found on this topic."
    
formatted_citations = []
for index, paper in enumerate(papers, 1):
    authors = ", ".join(paper['authors'][:3]) # Limit to top 3 authors
    if len(paper['authors']) > 3:
        authors += " et al."
        
    citation = (
        f"{index}. **[{paper['title']}]({paper['url']})**\n"
        f"   *Published:* {paper['published']} | *Authors:* {authors}\n"
        f"   *Snippet:* {paper['summary']}"
    )
    formatted_citations.append(citation)
    
# return "\n\n".join(formatted_citations)

In [9]:
md_text = pymupdf4llm.to_markdown("test_data/ccn_assignment.pdf", page_chunks=True)

In [10]:
[print(i) for i in md_text]

defaultdict(<function make_page_chunk.<locals>.<lambda> at 0x7038c9d05640>, {'metadata': {'format': 'PDF 1.6', 'title': 'CCN Assignment', 'author': '', 'subject': '', 'keywords': '', 'creator': 'Adobe Illustrator 24.3 (Windows)', 'producer': 'Adobe PDF library 15.00', 'creationDate': "D:20250831211904+06'00'", 'modDate': "D:20250831212100+05'00'", 'trapped': '', 'encryption': None, 'file_path': 'test_data/ccn_assignment.pdf', 'page_count': 5, 'page_number': 1}, 'toc_items': [], 'page_boxes': [{'index': 0, 'class': 'picture', 'bbox': (212, 134, 381, 304), 'pos': (0, 2)}, {'index': 1, 'class': 'section-header', 'bbox': (67, 350, 528, 455), 'pos': (2, 45)}, {'index': 2, 'class': 'section-header', 'bbox': (200, 473, 393, 497), 'pos': (45, 61)}, {'index': 3, 'class': 'section-header', 'bbox': (193, 576, 403, 600), 'pos': (61, 80)}, {'index': 4, 'class': 'text', 'bbox': (216, 630, 377, 655), 'pos': (80, 91)}], 'text': '\n\n# CS-327 COMPUTER COMMUNiCATiON NETWORKS \n\n## ASSiGNMENT \n\n## SUB

[None, None, None, None, None]

In [ ]:
for i in md_text:
    print(i["metadata"]["page_number"])
    print(i["text"])

In [12]:
documents = []
for page in md_text:
    doc = Document(
        metadata = {"page":page["metadata"]["page_number"]},
        page_content = page["text"]
    )
    # print(doc)
    documents.append(doc)

In [13]:
documents

[Document(metadata={'page': 1}, page_content='\n\n# CS-327 COMPUTER COMMUNiCATiON NETWORKS \n\n## ASSiGNMENT \n\n## SUBMiTTED BY: \n\nCS-23055 \n\n'),
 Document(metadata={'page': 2}, page_content='- Q1: Identify and illustrate the type of physical media used for your home internet connection (attach snapshot of the said connection as well). List all the communication links available within your home network setup. \n\n   1. The internet connection in my home coming from the ISP using uses a Cat5e Ethernet cable (RJ-45 connector) over twisted pair copper wiring which is plugged into the router. \n\nThen my PC also connects through a Cat5e Ethernet cable (RJ-45, twisted pair copper that runs from the router to the PC. \n\n\n\n<!-- Start of picture text -->\nTwisted Pair Cable<br>(Inside)<br>RJ-45 connector<br><!-- End of picture text -->\n\n\n\n2. Television Cable: The TV Cable in my home uses a coaxial cable which connects the service provider directly to the TV set-top box, delivering 

In [14]:
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
    ("####", "Header 4"),
]

In [15]:
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on,
    strip_headers=False
)

In [16]:
header_docs = []
for doc in documents:
    splits = markdown_splitter.split_text(doc.page_content)
    for s in splits:
        s.metadata.update(doc.metadata)
        header_docs.append(s)

In [17]:
header_docs

[Document(metadata={'Header 1': 'CS-327 COMPUTER COMMUNiCATiON NETWORKS', 'Header 2': 'ASSiGNMENT', 'page': 1}, page_content='# CS-327 COMPUTER COMMUNiCATiON NETWORKS  \n## ASSiGNMENT'),
 Document(metadata={'Header 1': 'CS-327 COMPUTER COMMUNiCATiON NETWORKS', 'Header 2': 'SUBMiTTED BY:', 'page': 1}, page_content='## SUBMiTTED BY:  \nCS-23055'),
 Document(metadata={'page': 2}, page_content='- Q1: Identify and illustrate the type of physical media used for your home internet connection (attach snapshot of the said connection as well). List all the communication links available within your home network setup.  \n1. The internet connection in my home coming from the ISP using uses a Cat5e Ethernet cable (RJ-45 connector) over twisted pair copper wiring which is plugged into the router.  \nThen my PC also connects through a Cat5e Ethernet cable (RJ-45, twisted pair copper that runs from the router to the PC.  \n<!-- Start of picture text -->\nTwisted Pair Cable<br>(Inside)<br>RJ-45 connect

In [18]:
text_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.MARKDOWN,
    chunk_size=1600,
    chunk_overlap=300
)

In [19]:
final_chunks = text_splitter.split_documents(header_docs)

In [20]:
final_chunks

[Document(metadata={'Header 1': 'CS-327 COMPUTER COMMUNiCATiON NETWORKS', 'Header 2': 'ASSiGNMENT', 'page': 1}, page_content='# CS-327 COMPUTER COMMUNiCATiON NETWORKS  \n## ASSiGNMENT'),
 Document(metadata={'Header 1': 'CS-327 COMPUTER COMMUNiCATiON NETWORKS', 'Header 2': 'SUBMiTTED BY:', 'page': 1}, page_content='## SUBMiTTED BY:  \nCS-23055'),
 Document(metadata={'page': 2}, page_content='- Q1: Identify and illustrate the type of physical media used for your home internet connection (attach snapshot of the said connection as well). List all the communication links available within your home network setup.  \n1. The internet connection in my home coming from the ISP using uses a Cat5e Ethernet cable (RJ-45 connector) over twisted pair copper wiring which is plugged into the router.  \nThen my PC also connects through a Cat5e Ethernet cable (RJ-45, twisted pair copper that runs from the router to the PC.  \n<!-- Start of picture text -->\nTwisted Pair Cable<br>(Inside)<br>RJ-45 connect

In [21]:
vector_store = Chroma.from_documents(
    embedding=embedding_model,
    documents=final_chunks,
    persist_directory="./chroma_db"
)

In [22]:
vector_retriever = vector_store.as_retriever(search_kwargs={"k": 25,})

In [23]:
bm25_retriever = BM25Retriever.from_documents(final_chunks)

In [24]:
bm25_retriever.k = 25

In [25]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.5, 0.5]
)

In [26]:
hf_cross_encoder_model = HuggingFaceCrossEncoder(
    model_name = "BAAI/bge-reranker-base"
)

Loading weights: 100%|██████████| 201/201 [00:01<00:00, 192.02it/s]


In [27]:
base_compressor = CrossEncoderReranker(model=hf_cross_encoder_model, top_n=10)

In [28]:
hybrid_rerank_retriever = ContextualCompressionRetriever(
    base_retriever=ensemble_retriever,
    base_compressor=base_compressor
)

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", 
     "You are a strict RAG assistant. You must NEVER use outside knowledge. "
     "If the context lacks the answer, you must output EXACTLY: "
     "'The given document does not contain context to this query.'"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human",
     "<context>\n{context}\n</context>\n\n"
     "Question: {input}\n\n"
     "STRICT INSTRUCTIONS:\n"
     "1. Answer ONLY using the facts in <context> above.\n"
     "2. If the context is empty or does not explicitly answer the question, reply EXACTLY with:\n"
     "   'The given document does not contain context to this query.'\n"
     "3. Do NOT use outside knowledge or make assumptions.\n"
     "4. Do NOT explain your reasoning. Provide the final answer immediately and concisely."
    )
])

In [30]:
parser = StrOutputParser()

In [31]:
mq_retriever = MultiQueryRetriever.from_llm(
    retriever=ensemble_retriever,
    llm=llm
)

In [32]:
final_hybrid_rerank_retriever = ContextualCompressionRetriever(
    base_compressor=base_compressor,
    base_retriever=mq_retriever
)

In [33]:
final_hybrid_rerank_retriever

ContextualCompressionRetriever(base_compressor=CrossEncoderReranker(model=HuggingFaceCrossEncoder(client=CrossEncoder(
  (0): Transformer({'transformer_task': 'sequence-classification', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'logits'}}, 'module_output_name': 'scores', 'architecture': 'XLMRobertaForSequenceClassification'})
), model_name='BAAI/bge-reranker-base', model_kwargs={}), top_n=10), base_retriever=MultiQueryRetriever(retriever=EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x70378d7c1160>, search_kwargs={'k': 25}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x70378d7c3a10>, k=25)], weights=[0.5, 0.5]), llm_chain=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='You are an AI language model assistant. Your task is\n    to generate 3 different versions of the given user\n    question to retrie

In [34]:
def formatter(docs):
    return "\n---\n new document \n---\n".join(doc.page_content for doc in docs)

In [35]:
def get_input(x):
    return x["input"]

def get_history(x):
    return x["chat_history"]

In [36]:
context_rephrasing_prompt = ChatPromptTemplate([
    ("system", "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."),

    MessagesPlaceholder(variable_name="chat_history"),

    ("human", "{input}")
])

In [37]:
history_aware_retriever = create_history_aware_retriever(
    llm = llm,
    retriever=final_hybrid_rerank_retriever,
    prompt = context_rephrasing_prompt
)

In [60]:
ai_response_summary_prompt = ChatPromptTemplate.from_template(
    "Condense the following text into a brief, single-sentence summary of the core facts. \n"
    "CRITICAL RULES:\n"
    "1. Do NOT add outside knowledge.\n"
    "2. Do NOT add intro or outro.\n"
    "2. If the text is a short list or under 2 sentences, return it exactly as is.\n\n"
    "Text to summarize:\n{response}"
)

In [61]:
summary_chain = ai_response_summary_prompt | llm | parser

In [62]:
chain = {
    "context": history_aware_retriever | formatter,
    "chat_history": get_history,
    "input": get_input
} | prompt | llm | parser

In [68]:
chat_history = []
chat_history

[]

In [69]:
query = "What type of physical media is used for the television cable, and what does it connect?"

full_response = ""
chunks = chain.stream({"input": query, "chat_history": chat_history})
for chunk in chunks:
    print(chunk, flush=True, end="")
    full_response += chunk

fallback_phrase = "The given document does not contain context to this query."

if fallback_phrase in full_response:
    ai_summary = fallback_phrase 
    print(f"\n---\nAI Summary Bypassed: {ai_summary}\n---\n")
else:
    ai_summary = summary_chain.invoke({"response": full_response})
    print(f"\n---\nAI Summary: {ai_summary}\n---\n")


chat_history.extend([
    HumanMessage(content=query),
    AIMessage(content=ai_summary)
])

chat_history = chat_history[-10:]

print(chat_history)

Coaxial cable connects the service provider directly to the TV set-top box, delivering digital TV channels.
---
AI Summary: Coaxial cables connect the service provider directly to the TV set-top box, delivering digital TV channels.
---

[HumanMessage(content='What type of physical media is used for the television cable, and what does it connect?', additional_kwargs={}, response_metadata={}), AIMessage(content='Coaxial cables connect the service provider directly to the TV set-top box, delivering digital TV channels.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


In [70]:
query = "According to Q3, what is the starting price and RAM capacity of the ThinkCentre M70t Gen 6?"

full_response = ""
chunks = chain.stream({"input": query, "chat_history": chat_history})
for chunk in chunks:
    print(chunk, flush=True, end="")
    full_response += chunk

fallback_phrase = "The given document does not contain context to this query."

if fallback_phrase in full_response:
    ai_summary = fallback_phrase 
    print(f"\n---\nAI Summary Bypassed: {ai_summary}\n---\n")
else:
    ai_summary = summary_chain.invoke({"response": full_response})
    print(f"\n---\nAI Summary: {ai_summary}\n---\n")


chat_history.extend([
    HumanMessage(content=query),
    AIMessage(content=ai_summary)
])

chat_history = chat_history[-10:]

print(chat_history)

ThinkCentre M70t Gen 6: 
 Processor - Intel Core Ultra 7 265 vPro 
Memory (RAM) - 4 x UDIMM Slots supporting up to 128 GB
---
AI Summary: The ThinkCentre M70t Gen 6 features a processor with an Intel Core Ultra 7 265 vPro and supports up to 128 GB of RAM.
---

[HumanMessage(content='What type of physical media is used for the television cable, and what does it connect?', additional_kwargs={}, response_metadata={}), AIMessage(content='Coaxial cables connect the service provider directly to the TV set-top box, delivering digital TV channels.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='According to Q3, what is the starting price and RAM capacity of the ThinkCentre M70t Gen 6?', additional_kwargs={}, response_metadata={}), AIMessage(content='The ThinkCentre M70t Gen 6 features a processor with an Intel Core Ultra 7 265 vPro and supports up to 128 GB of RAM.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_to

In [71]:
query = "List the five network services explained in the document."

full_response = ""
chunks = chain.stream({"input": query, "chat_history": chat_history})
for chunk in chunks:
    print(chunk, flush=True, end="")
    full_response += chunk

fallback_phrase = "The given document does not contain context to this query."

if fallback_phrase in full_response:
    ai_summary = fallback_phrase 
    print(f"\n---\nAI Summary Bypassed: {ai_summary}\n---\n")
else:
    ai_summary = summary_chain.invoke({"response": full_response})
    print(f"\n---\nAI Summary: {ai_summary}\n---\n")


chat_history.extend([
    HumanMessage(content=query),
    AIMessage(content=ai_summary)
])

chat_history = chat_history[-10:]

print(chat_history)

HTTP/HTTPS
Cloud Storage
Security Services
iPTV/Streaming Service 
VPN Service
---
AI Summary: The core facts are: HTTP/HTTPS, Cloud Storage, Security Services, iPTV/Streaming Service, and VPN Service.
---

[HumanMessage(content='What type of physical media is used for the television cable, and what does it connect?', additional_kwargs={}, response_metadata={}), AIMessage(content='Coaxial cables connect the service provider directly to the TV set-top box, delivering digital TV channels.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='According to Q3, what is the starting price and RAM capacity of the ThinkCentre M70t Gen 6?', additional_kwargs={}, response_metadata={}), AIMessage(content='The ThinkCentre M70t Gen 6 features a processor with an Intel Core Ultra 7 265 vPro and supports up to 128 GB of RAM.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='List the five netw

In [72]:
query = "Explain sycophantic behaviour of AI."

full_response = ""
chunks = chain.stream({"input": query, "chat_history": chat_history})
for chunk in chunks:
    print(chunk, flush=True, end="")
    full_response += chunk

fallback_phrase = "The given document does not contain context to this query."

if fallback_phrase in full_response:
    ai_summary = fallback_phrase 
    print(f"\n---\nAI Summary Bypassed: {ai_summary}\n---\n")
else:
    ai_summary = summary_chain.invoke({"response": full_response})
    print(f"\n---\nAI Summary: {ai_summary}\n---\n")


chat_history.extend([
    HumanMessage(content=query),
    AIMessage(content=ai_summary)
])

chat_history = chat_history[-10:]

print(chat_history)

The given document does not contain context to this query.
---
AI Summary Bypassed: The given document does not contain context to this query.
---

[HumanMessage(content='What type of physical media is used for the television cable, and what does it connect?', additional_kwargs={}, response_metadata={}), AIMessage(content='Coaxial cables connect the service provider directly to the TV set-top box, delivering digital TV channels.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='According to Q3, what is the starting price and RAM capacity of the ThinkCentre M70t Gen 6?', additional_kwargs={}, response_metadata={}), AIMessage(content='The ThinkCentre M70t Gen 6 features a processor with an Intel Core Ultra 7 265 vPro and supports up to 128 GB of RAM.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='List the five network services explained in the document.', additional_kwargs

In [73]:
query = "What is the exact Wi-Fi speed of the router mentioned in the Q4 host devices list?"

full_response = ""
chunks = chain.stream({"input": query, "chat_history": chat_history})
for chunk in chunks:
    print(chunk, flush=True, end="")
    full_response += chunk

fallback_phrase = "The given document does not contain context to this query."

if fallback_phrase in full_response:
    ai_summary = fallback_phrase 
    print(f"\n---\nAI Summary Bypassed: {ai_summary}\n---\n")
else:
    ai_summary = summary_chain.invoke({"response": full_response})
    print(f"\n---\nAI Summary: {ai_summary}\n---\n")


chat_history.extend([
    HumanMessage(content=query),
    AIMessage(content=ai_summary)
])

chat_history = chat_history[-10:]

print(chat_history)

The given document does not contain information about Wi-Fi speed of the router mentioned in the Q4 host devices list.
---
AI Summary: There is no information about Wi-Fi speed of the router in the given document.
---

[HumanMessage(content='What type of physical media is used for the television cable, and what does it connect?', additional_kwargs={}, response_metadata={}), AIMessage(content='Coaxial cables connect the service provider directly to the TV set-top box, delivering digital TV channels.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='According to Q3, what is the starting price and RAM capacity of the ThinkCentre M70t Gen 6?', additional_kwargs={}, response_metadata={}), AIMessage(content='The ThinkCentre M70t Gen 6 features a processor with an Intel Core Ultra 7 265 vPro and supports up to 128 GB of RAM.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='List t

In [74]:
query = "What is the maximum storage capacity for the HP OmniDesk Desktop?"

full_response = ""
chunks = chain.stream({"input": query, "chat_history": chat_history})
for chunk in chunks:
    print(chunk, flush=True, end="")
    full_response += chunk

fallback_phrase = "The given document does not contain context to this query."

if fallback_phrase in full_response:
    ai_summary = fallback_phrase 
    print(f"\n---\nAI Summary Bypassed: {ai_summary}\n---\n")
else:
    ai_summary = summary_chain.invoke({"response": full_response})
    print(f"\n---\nAI Summary: {ai_summary}\n---\n")


chat_history.extend([
    HumanMessage(content=query),
    AIMessage(content=ai_summary)
])

chat_history = chat_history[-10:]

print(chat_history)

128 GB is the maximum RAM capacity mentioned for any Desktop in the document. The provided details do not specify an exact storage limit for HP OmniDesk Desktop.
---
AI Summary: The maximum RAM capacity for the HP OmniDesk Desktop mentioned in the document is 128 GB, but no specific storage limit is specified.
---

[HumanMessage(content='What type of physical media is used for the television cable, and what does it connect?', additional_kwargs={}, response_metadata={}), AIMessage(content='Coaxial cables connect the service provider directly to the TV set-top box, delivering digital TV channels.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='According to Q3, what is the starting price and RAM capacity of the ThinkCentre M70t Gen 6?', additional_kwargs={}, response_metadata={}), AIMessage(content='The ThinkCentre M70t Gen 6 features a processor with an Intel Core Ultra 7 265 vPro and supports up to 128 GB of RAM.', additional_kw

In [75]:
query = "What processor does the Dell PowerEdge T160 Tower Server use?"

full_response = ""
chunks = chain.stream({"input": query, "chat_history": chat_history})
for chunk in chunks:
    print(chunk, flush=True, end="")
    full_response += chunk

fallback_phrase = "The given document does not contain context to this query."

if fallback_phrase in full_response:
    ai_summary = fallback_phrase 
    print(f"\n---\nAI Summary Bypassed: {ai_summary}\n---\n")
else:
    ai_summary = summary_chain.invoke({"response": full_response})
    print(f"\n---\nAI Summary: {ai_summary}\n---\n")


chat_history.extend([
    HumanMessage(content=query),
    AIMessage(content=ai_summary)
])

chat_history = chat_history[-10:]

print(chat_history)

One Intel Xeon 6300 or Xeon E-2400 series, or Pentium G7400 processor.
---
AI Summary: A single Intel processor (Xeon 6300/Xeon E-2400 or Pentium G7400) is required.
---

[HumanMessage(content='What type of physical media is used for the television cable, and what does it connect?', additional_kwargs={}, response_metadata={}), AIMessage(content='Coaxial cables connect the service provider directly to the TV set-top box, delivering digital TV channels.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='According to Q3, what is the starting price and RAM capacity of the ThinkCentre M70t Gen 6?', additional_kwargs={}, response_metadata={}), AIMessage(content='The ThinkCentre M70t Gen 6 features a processor with an Intel Core Ultra 7 265 vPro and supports up to 128 GB of RAM.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='List the five network services explained in the docume

In [76]:
query = "And how much does it cost?"

full_response = ""
chunks = chain.stream({"input": query, "chat_history": chat_history})
for chunk in chunks:
    print(chunk, flush=True, end="")
    full_response += chunk

fallback_phrase = "The given document does not contain context to this query."

if fallback_phrase in full_response:
    ai_summary = fallback_phrase 
    print(f"\n---\nAI Summary Bypassed: {ai_summary}\n---\n")
else:
    ai_summary = summary_chain.invoke({"response": full_response})
    print(f"\n---\nAI Summary: {ai_summary}\n---\n")


chat_history.extend([
    HumanMessage(content=query),
    AIMessage(content=ai_summary)
])

chat_history = chat_history[-10:]

print(chat_history)

Coaxial cables cost $0 and connect the service provider directly to the TV set-top box, delivering digital TV channels.
---
AI Summary: Coaxial cables cost $0 and connect the service provider to the TV set-top box to deliver digital TV channels.
---

[HumanMessage(content='What type of physical media is used for the television cable, and what does it connect?', additional_kwargs={}, response_metadata={}), AIMessage(content='Coaxial cables connect the service provider directly to the TV set-top box, delivering digital TV channels.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='According to Q3, what is the starting price and RAM capacity of the ThinkCentre M70t Gen 6?', additional_kwargs={}, response_metadata={}), AIMessage(content='The ThinkCentre M70t Gen 6 features a processor with an Intel Core Ultra 7 265 vPro and supports up to 128 GB of RAM.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[